In [1]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
import json
import collections
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import zipfile

from tqdm import tqdm
import shutil

np.random.seed(42)

In [2]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [3]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [4]:
from py.utils import verifyDir,verifyFile, verifyDataFrame

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
DATA_PATH, MODEL_PATH

('/media/felipe/DATA19/datasets/', '/media/felipe/DATA19/models/')

In [8]:
ML_TASK = "classifications"
PRECEPTION_METRIC = "safety"

SEG_MODEL_NAME="OneFormer_Swin_Large"
SEG_DATASET="ade20k" # cityscapes

USE_UPD=True
FILTER_FEATURES=False
BY_GROUPS=False
BINARIZE_FEATURES=False

In [9]:
QSCORE_PATH=f"{DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{DATA_PATH}pp2/images/"

In [ ]:
OUT_NAME = f"{PRECEPTION_METRIC}"
OUT_NAME += "_filter" if FILTER_FEATURES else ""
OUT_NAME += "_bin" if BINARIZE_FEATURES else ""
OUT_NAME

In [12]:
CF_DIR = f"{MODEL_PATH}"
CF_DIR += f"{SEG_DATASET}_upd4k" if USE_UPD else f"{SEG_DATASET}"
CF_DIR += "_group/" if BY_GROUPS else "/"
CF_DIR += "" if USE_UPD else f"{SEG_MODEL_NAME}/"
CF_DIR += f"counterfactuals/{OUT_NAME}/"

### Loading Data

In [15]:
cf_test_data_df = pd.read_csv(f"{CF_DIR}cf_test_data.csv", sep=";", low_memory=False)

In [ ]:
features_name = cf_test_data_df.iloc[:, 1:-1].columns.tolist()

In [17]:
label_map = dict( zip( cf_test_data_df['target'], cf_test_data_df['label'] ) )
label_map

{1: 'safety', 0: 'not safety'}

# LLM Interpretations

In [18]:
from py.counterfactuals import CounterfactualAnalyzer

In [21]:
cf_analyzer = CounterfactualAnalyzer()
cf_analyzer.load(CF_DIR)

In [ ]:
results = cf_analyzer.get_results()
unsafe2safe_df = results["nearest_cfs"].copy()
unsafe2safe_df.sort_values(by="diff_prob", ascending=False, inplace=True)
unsafe2safe_df.tail(30)

In [ ]:
os.exit()

# LLM Interpretations

In [ ]:
# Pixel ratio case
visual_elements = ["tree", "graffiti", "streetlight", "broken sidewalk", "car"]

x = [0.10, 0.15, 0.00, 0.08, 0.20]   # Original pixel ratios
x_prime = [0.25, 0.02, 0.03, 0.00, 0.18]  # Counterfactual pixel ratios

epsilon = 0.02  # Threshold for significance

added = []
removed = []

for i in range(len(visual_elements)):
    delta = x_prime[i] - x[i]
    if delta > epsilon:
        added.append(f"{visual_elements[i]}: increased from {int(x[i]*100)}% to {int(x_prime[i]*100)}%")
    elif delta < -epsilon:
        removed.append(f"{visual_elements[i]}: decreased from {int(x[i]*100)}% to {int(x_prime[i]*100)}%")

# Create prompt
prompt_pixel = f"""
You are an expert in urban planning and visual perception. Your task is to explain how changes in specific visual elements of a street scene might influence the way people perceive safety.

These changes come from comparing an original image with a counterfactual version.

[Added elements]:
{chr(10).join(added) if added else 'None'}

[Removed elements]:
{chr(10).join(removed) if removed else 'None'}

Please write a concise explanation (2–4 sentences) describing how these changes may affect safety perception. Focus on order, comfort, and social signals.
"""
print(prompt_pixel)

# Binary case: presence (1) or absence (0)
visual_elements = ["tree", "graffiti", "streetlight", "broken sidewalk", "car"]

x = [0, 1, 0, 1, 1]       # Original
x_prime = [1, 0, 1, 0, 1] # Counterfactual

# Calculate difference
delta = [x_p - x_o for x_o, x_p in zip(x, x_prime)]

added = [visual_elements[i] for i, d in enumerate(delta) if d == 1]
removed = [visual_elements[i] for i, d in enumerate(delta) if d == -1]

# Create prompt
prompt_binary = f"""
You are an expert in urban planning and visual perception. Your task is to explain how changes in specific visual elements of a street scene might influence the way people perceive safety.

These changes come from comparing an original image with a counterfactual version.

[Added elements]: {', '.join(added) if added else 'None'}
[Removed elements]: {', '.join(removed) if removed else 'None'}

Please write a concise explanation (2–4 sentences) describing how these changes may affect safety perception. Focus on order, comfort, and social signals.
"""
print(prompt_binary)